# EfficientNetV2S - KHOTAA Diabetic Foot Ulcer Classification

## 1. Imports & Configuration

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import numpy as np
from sklearn.model_selection import StratifiedKFold

sys.path.append('../')
sys.path.append('./')

from dataset_loader import SplitFolderDatasetLoader
from dataset_preprocessing import DFUPreprocessing
from utils.checkpoint_manager import CheckpointManager
from utils.training_engine import TrainingEngine, create_optimizer
from utils.metrics_evaluator import (
    calculate_metrics, print_metrics, plot_confusion_matrix,
    plot_roc_curve, plot_training_history
)

print("Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load Dataset

In [ ]:
# Load dataset
loader = SplitFolderDatasetLoader(root_dir='../../dataset')
classes = loader.get_classes()
num_classes = loader.get_num_classes()

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

# Initialize preprocessing
preprocessor = DFUPreprocessing()
train_transform = preprocessor.get_train_transforms()
val_test_transform = preprocessor.get_valid_test_transforms()

# Dataset class
class DFUDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Prepare data for cross-validation
X_train, y_train = loader.load_split_paths('train', shuffle=True)
X_val, y_val = loader.load_split_paths('valid')
X_all = np.concatenate([X_train, X_val])
y_all = np.concatenate([y_train, y_val])

# Test set (untouched until final evaluation)
X_test, y_test = loader.load_split_paths('test')
test_dataset = DFUDataset(X_test, y_test, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Initialize 5-fold stratified cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nTotal training samples (train+valid): {len(X_all)}")
print(f"Test samples: {len(X_test)}")
print("Dataset loaded and ready for 5-fold cross-validation")

## 3. Model Definition

In [ ]:
# Setup device and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")

# Create EfficientNetV2S model with best implementation practices
def create_efficientnet_model(num_classes=4, pretrained=True, freeze_backbone=False):
    """
    Create EfficientNetV2S model for DFU classification with best practices.
    
    EfficientNetV2S Architecture:
    - Improved mobile inverted bottleneck (MBConv) with new activations (SiLU)
    - Progressive training compatibility (gradual image size increase)
    - Better scaling coefficients (depth, width, resolution)
    - Stochastic depth for regularization
    - Squeeze-and-excitation blocks for channel attention
    - 21.6M parameters (optimized for balance of accuracy and efficiency)
    
    Args:
        num_classes: Number of output classes (4 for DFU grades)
        pretrained: Use ImageNet pretrained weights (IMAGENET1K_V1)
        freeze_backbone: Freeze feature extraction layers (True for limited data)
    
    Returns:
        EfficientNetV2S model configured for DFU classification
    
    Reference: Tan & Le, 2021 - EfficientNetV2: Smaller Models and Faster Training
    """
    
    # Load pretrained or random initialization
    if pretrained:
        model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        print("Loaded pretrained ImageNet weights (IMAGENET1K_V1)")
    else:
        model = models.efficientnet_v2_s(weights=None)
        print("Using random initialization")
    
    # Get feature extraction dimension
    # EfficientNetV2S output: 1280 features before classifier
    num_features = model.classifier[1].in_features
    
    # Modify final classifier layer for 4-class DFU classification
    # Original: Linear(1280 -> 1000)
    # Modified: Linear(1280 -> num_classes)
    model.classifier[1] = nn.Linear(num_features, num_classes)
    
    # Transfer learning strategy: Freeze backbone if training data is limited
    if freeze_backbone and pretrained:
        for param in model.features.parameters():
            param.requires_grad = False
        print("Backbone frozen - only classifier will be trained")
        # Count trainable parameters
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Trainable parameters: {trainable_params:,} / Total: {total_params:,} ({trainable_params/total_params*100:.2f}%)")
    else:
        # Full fine-tuning: all layers trainable
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Fine-tuning mode - All parameters trainable: {trainable_params:,}")
        print(f"Total parameters: {total_params:,}")
    
    return model

# Test model creation and display statistics
print("\n" + "="*60)
print("MODEL CONFIGURATION & STATISTICS")
print("="*60)

test_model = create_efficientnet_model(num_classes=num_classes, pretrained=True, freeze_backbone=False)

# Display model architecture summary
print(f"\nModel Architecture:")
print(f"  - Name: EfficientNetV2S")
print(f"  - Input size: 224x224 (RGB)")
print(f"  - Output classes: {num_classes}")
print(f"  - Feature dimension: 1280")
print(f"  - Backbone: EfficientNetV2 (MBConv + attention + stochastic depth)")
print(f"  - Classifier: Linear({1280} -> {num_classes})")

# Layer information
print(f"\nModel Layers:")
print(f"  - Feature extraction blocks: {len(list(test_model.features))}")
print(f"  - Classifier layers: 2 (Linear + Dropout)")

# Detailed layer info
print(f"\nClassifier Details:")
print(f"  Layer 1 (Dropout): {test_model.classifier[0]}")
print(f"  Layer 2 (Linear): {test_model.classifier[1]}")

print(f"\n" + "="*60)


## 4. Training

In [ ]:
# 5-Fold Cross-Validation Training
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_all, y_all), 1):
    print(f"\n{'='*60}\nFOLD {fold}/5\n{'='*60}")
    
    # Prepare fold data
    X_train_fold = [X_all[i] for i in train_idx]
    y_train_fold = y_all[train_idx]
    X_val_fold = [X_all[i] for i in val_idx]
    y_val_fold = y_all[val_idx]
    
    train_dataset = DFUDataset(X_train_fold, y_train_fold, transform=train_transform)
    val_dataset = DFUDataset(X_val_fold, y_val_fold, transform=val_test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Create model
    model = create_efficientnet_model(num_classes=num_classes, pretrained=True)
    model = model.to(device)
    
    # Setup optimizer using helper function (SGD with momentum=0.8)
    optimizer = create_optimizer(model, lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'efficientnetv2s_fold{fold}')
    engine = TrainingEngine(model=model, device=device)
    
    # Train
    history = engine.train(
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=30,
        scheduler=scheduler,
        checkpoint_manager=checkpoint_manager,
        early_stopping_patience=7,
        use_early_stopping=True,
        verbose=True
    )
    
    # Store results
    best_val_acc = max(history['val_acc'])
    fold_results.append({
        'fold': fold,
        'best_val_acc': best_val_acc,
        'final_val_acc': history['val_acc'][-1],
        'stopped_epoch': history['stopped_epoch'],
        'history': history,
        'checkpoint_manager': checkpoint_manager
    })
    print(f"Fold {fold} - Best Acc: {best_val_acc*100:.2f}% (stopped at epoch {history['stopped_epoch']})")

# Cross-validation summary
avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
std_acc = np.std([r['best_val_acc'] for r in fold_results])
avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])

print(f"\n{'='*60}")
print(f"5-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Average Epochs: {avg_epochs:.1f}")
print(f"\nIndividual Fold Results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
print(f"{'='*60}")

## 5. Evaluation & Plots

In [ ]:
# Test Set Evaluation
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

# Check if results already exist
import os
import json

# Organize results by model name
model_results_dir = 'results/efficientnetv2s'
results_file = f'{model_results_dir}/efficientnetv2s_results.json'

if os.path.exists(results_file):
    print(f"\nFound existing results file: {results_file}")
    print("Loading previously saved results...\n")
    
    with open(results_file, 'r') as f:
        efficientnet_results = json.load(f)
    
    # Display the loaded results
    print("="*60)
    print("EFFICIENTNETV2S - LOADED RESULTS")
    print("="*60)
    
    cv_results = efficientnet_results['cv_results']
    test_results = efficientnet_results['test_results']
    inference = efficientnet_results.get('inference_time', {})
    
    print(f"\nCROSS-VALIDATION RESULTS:")
    print(f"   Mean Accuracy: {cv_results['val_accuracy']['mean']*100:.2f}% ± {cv_results['val_accuracy']['std']*100:.2f}%")
    print(f"   Average Epochs: {cv_results['avg_epochs']:.1f}")
    
    print(f"\nTEST SET RESULTS:")
    print(f"   Test Accuracy: {test_results['test_accuracy']*100:.2f}%")
    print(f"   Test Loss: {test_results.get('test_loss', 'N/A'):.4f}" if 'test_loss' in test_results else "   Test Loss: N/A")
    print(f"   Precision: {test_results['precision']:.4f}")
    print(f"   Recall: {test_results['recall']:.4f}")
    print(f"   F1-Score: {test_results['f1_score']:.4f}")
    print(f"   Specificity: {test_results.get('specificity', 'N/A'):.4f}" if 'specificity' in test_results else "   Specificity: N/A")
    print(f"   Sensitivity: {test_results.get('sensitivity', 'N/A'):.4f}" if 'sensitivity' in test_results else "   Sensitivity: N/A")
    print(f"   MCC: {test_results['mcc']:.4f}")
    print(f"   AUC: {test_results['auc']:.4f}")
    
    if inference:
        print(f"\nINFERENCE TIME:")
        print(f"   Avg Time/Image: {inference.get('avg_time_per_image_ms', 'N/A'):.2f}ms")
        print(f"   Throughput: {inference.get('throughput_fps', 'N/A'):.1f} images/second")
    
    print(f"\nINDIVIDUAL FOLD RESULTS:")
    for fold_result in cv_results['fold_results']:
        print(f"   Fold {fold_result['fold']}: {fold_result['best_val_acc']*100:.2f}% (epoch {fold_result['stopped_epoch']})")
    
    print("="*60)
    
    # Check for visualization files
    viz_files = [
        f'{model_results_dir}/confusion_matrix.png',
        f'{model_results_dir}/roc_curve.png',
        f'{model_results_dir}/training_history.png'
    ]
    
    print(f"\nVISUALIZATION FILES:")
    for viz_file in viz_files:
        if os.path.exists(viz_file):
            print(f"   {viz_file}")
        else:
            print(f"   {viz_file} (not found)")
    
    print(f"\nTIP: To regenerate results, delete {results_file} and re-run training")
    print("="*60)

else:
    # Original evaluation code - runs only if results don't exist
    print(f"\nNo existing results found at {results_file}")
    print("Running full evaluation...\n")
    
    # Load best fold model
    best_fold_idx = np.argmax([r['best_val_acc'] for r in fold_results])
    best_fold_num = fold_results[best_fold_idx]['fold']

    print(f"Loading best model from Fold {best_fold_num}")

    checkpoint_manager = fold_results[best_fold_idx]['checkpoint_manager']
    
    # Load best model using the checkpoint manager
    model = checkpoint_manager.load_best_model(
        fold_index=0,
        create_model_fn=lambda: create_efficientnet_model(num_classes=num_classes, pretrained=False),
        metric_name='val_acc'
    )
    model = model.to(device)

    engine = TrainingEngine(model=model, device=device)

    # Evaluate with inference time tracking
    test_loss, test_acc, predictions, true_labels, inference_time = engine.evaluate(
        test_loader, 
        criterion, 
        measure_inference_time=True
    )

    print(f"\nTest Accuracy: {test_acc*100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"\nInference Time Statistics:")
    print(f"  Total Time: {inference_time['total_time']:.4f}s")
    print(f"  Avg Time/Batch: {inference_time['avg_time_per_batch']*1000:.2f}ms ± {inference_time['std_time_per_batch']*1000:.2f}ms")
    print(f"  Avg Time/Image: {inference_time['avg_time_per_image']*1000:.2f}ms")
    print(f"  Throughput: {inference_time['images_per_second']:.1f} images/second")

    # Get probabilities for AUC
    model.eval()
    all_probs = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs.to(device))
            probs = torch.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())

    y_pred_proba = np.vstack(all_probs)

    # Calculate all metrics
    metrics = calculate_metrics(
        y_true=true_labels,
        y_pred=predictions,
        y_pred_proba=y_pred_proba,
        class_names=classes,
        average='macro'
    )

    print("\n" + "="*60)
    print_metrics(metrics, title="EfficientNetV2S Test Results")
    print("="*60)

    # Save to results/efficientnetv2s folder
    os.makedirs(model_results_dir, exist_ok=True)

    # Confusion Matrix
    plot_confusion_matrix(
        y_true=true_labels,
        y_pred=predictions,
        class_names=classes,
        normalize=True,
        save_path=f'{model_results_dir}/confusion_matrix.png'
    )
    print(f"\nConfusion matrix saved to {model_results_dir}/confusion_matrix.png")

    # ROC Curve
    plot_roc_curve(
        y_true=true_labels,
        y_pred_proba=y_pred_proba,
        class_names=classes,
        save_path=f'{model_results_dir}/roc_curve.png'
    )
    print(f"ROC curve saved to {model_results_dir}/roc_curve.png")

    # Training History (best fold)
    plot_training_history(
        fold_results[best_fold_idx]['history'],
        save_path=f'{model_results_dir}/training_history.png'
    )
    print(f"Training history saved to {model_results_dir}/training_history.png")

    # Summary for Model Comparison
    print("\n" + "="*60)
    print("SUMMARY FOR MODEL COMPARISON")
    print("="*60)
    print(f"Model: EfficientNetV2S")
    print(f"Cross-Validation Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"Test Accuracy: {test_acc*100:.2f}%")
    print(f"Test F1-Score: {metrics['f1_score']:.4f}")
    print(f"Test MCC: {metrics['mcc']:.4f}")
    print(f"Test AUC: {metrics['auc']:.4f}")
    print(f"Average Training Epochs: {avg_epochs:.1f}")
    print(f"Inference Time: {inference_time['avg_time_per_image']*1000:.2f}ms per image")
    print(f"Throughput: {inference_time['images_per_second']:.1f} images/second")
    print("="*60)

In [ ]:
# Save results for model comparison
import json
import os

# Organize by model name
model_results_dir = 'results/efficientnetv2s'
results_file = f'{model_results_dir}/efficientnetv2s_results.json'

if os.path.exists(results_file):
    print(f"Results already saved at {results_file}")
    print(" To regenerate, delete the file and re-run training & evaluation")
else:
    # Prepare results dictionary
    efficientnet_results = {
        'model_name': 'EfficientNetV2S',
        'cv_results': {
            'val_accuracy': {'mean': float(avg_acc), 'std': float(std_acc)},
            'avg_epochs': float(avg_epochs),
            'fold_results': [
                {
                    'fold': r['fold'],
                    'best_val_acc': float(r['best_val_acc']),
                    'stopped_epoch': int(r['stopped_epoch'])
                }
                for r in fold_results
            ]
        },
        'test_results': {
            'test_accuracy': float(test_acc),
            'test_loss': float(test_loss),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'f1_score': float(metrics['f1_score']),
            'specificity': float(metrics['specificity']),
            'sensitivity': float(metrics['sensitivity']),
            'mcc': float(metrics['mcc']),
            'auc': float(metrics['auc'])
        },
        'inference_time': {
            'total_time': float(inference_time['total_time']),
            'avg_time_per_image_ms': float(inference_time['avg_time_per_image'] * 1000),
            'throughput_fps': float(inference_time['images_per_second'])
        }
    }

    # Save to JSON in model-specific directory
    os.makedirs(model_results_dir, exist_ok=True)
    with open(results_file, 'w') as f:
        json.dump(efficientnet_results, f, indent=4)

    print(f"Results saved to {results_file}")
    print("\nThese results can be used with the ModelComparison utility:")
    print("from utils.model_comparison import ModelComparison")
    print("comparison = ModelComparison()")
    print("comparison.add_model_result(**efficientnet_results)")
    print("\nEfficientNetV2S training complete!")

## 6. Save Results for Model Comparison

Save the results for later comparison with other models (ResNet50, ResNet101, MobileNet, DenseNet, GoogLeNet, PFCNN+DRNN).